In [1]:
import pandas as pd

# extraction date
date = "2019-07-20"

# input template
df_template = pd.read_excel("timeseries.xlsx")

# load pv and load data
pv_data = pd.read_csv(r"C:\Users\babuk\Repository\Tools_Dataset_Preparation\output\2019_15min_PV_Generation_data_3kW.csv")
load_data = pd.read_csv(r"C:\Users\babuk\Repository\Tools_Dataset_Preparation\output\2019_15min_load_profile_8kW.csv")
price_data = pd.read_csv(r"C:\Users\babuk\Repository\Tools_Dataset_Preparation\output\2019_15min_prosumer_price_curve.csv")

pv_data.index = pd.to_datetime(pv_data['datetime'], format='mixed', dayfirst=True)
load_data.index = pd.to_datetime(load_data['datetime'], format='mixed', dayfirst=True)
price_data = price_data.set_index(pd.date_range(start=date, end=date + " 23:45:00", freq='15min'))

# extract data for the given date
pv_series = pv_data.loc[pv_data.index.date == pd.to_datetime(date).date()].loc[:, 'PV_Power_kW']
pv_series = pv_series/3000 
pv_series = pv_series.reindex(pd.date_range(start=date, end=date + " 23:45:00", freq='15min'), fill_value=0)
load_series = load_data.loc[load_data.index.date == pd.to_datetime(date).date()].loc[:, 'h0_dyn']

# create a copy of the template dataframe
df_filled = df_template.copy()
df_filled.index = pd.date_range(start=date, end=date + " 23:45:00", freq='15min')
df_filled['solar_power'] = pv_series
df_filled['load_elec'] = load_series
df_filled['ele_price_in'] = price_data['Price']
df_filled['ele_price_out'] = 0.0786
df_filled = df_filled.round(5)


Download data

In [2]:
# save the filled dataframe to a new excel file
df_filled.to_excel(f"{date}_timeseries_deterministic.xlsx", index=False)